# Laboratorio L15 - Apprendimento non supervisionato: k-means

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessari: `pinguini.csv`, `L15_tramonto.npy` (immagine sintetica creata per il corso).

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## 1. k-means a mano (esercizio 1)

Sei punti e due centroidi iniziali. Un passo del k-means:

1. assegnare ogni punto al centroide più vicino
2. spostare ogni centroide nella media dei punti assegnati

- `np.argmin`: posizione del valore minimo

In [ ]:
punti = np.array([[1, 1], [1.5, 2], [2, 1], [6, 5], [7, 6], [6.5, 7]])
centroidi = np.array([[1.5, 3.0], [6.0, 0.0]])

for passo in range(3):
    distanze = np.sqrt(((punti[:, None, :] - centroidi[None, :, :]) ** 2).sum(axis=2))   # punti x centroidi
    gruppo = np.argmin(distanze, axis=1)
    print(f"passo {passo}: gruppi {gruppo}")
    centroidi = np.array([punti[gruppo == k].mean(axis=0) for k in range(2)])
    print("         nuovi centroidi", centroidi.round(2))

**Domanda**: dopo quanti passi i gruppi smettono di cambiare? Verificare a mano il primo passo.

Risposta:

## 2. k-means sui pinguini, senza la specie (esercizio 2)

Si usano solo lunghezza e profondità del becco, standardizzate. La specie non viene data al modello: si usa solo alla fine, per confrontare.

- `KMeans(n_clusters=3, n_init=10, random_state=0)`: 3 gruppi; `n_init=10` ripete l'algoritmo 10 volte da centroidi iniziali diversi e tiene il risultato migliore
- `fit(X)`: senza etichette
- `labels_`: il gruppo assegnato a ogni pinguino (0, 1, 2: numeri senza significato)

In [ ]:
pinguini = pd.read_csv("pinguini.csv").dropna(subset=["becco_lunghezza_mm", "becco_profondita_mm"])
X = StandardScaler().fit_transform(pinguini[["becco_lunghezza_mm", "becco_profondita_mm"]])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
pinguini["gruppo"] = km.labels_
pd.crosstab(pinguini["specie"], pinguini["gruppo"])

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(11, 4))
for s, c in zip(["Adelie", "Chinstrap", "Gentoo"], ["#2a78d6", "#eb6834", "#1baf7a"]):
    d = pinguini[pinguini["specie"] == s]
    assi[0].scatter(d["becco_lunghezza_mm"], d["becco_profondita_mm"], s=12, color=c, label=s)
assi[0].set_title("specie"); assi[0].legend()
assi[1].scatter(pinguini["becco_lunghezza_mm"], pinguini["becco_profondita_mm"], s=12, c=pinguini["gruppo"], cmap="viridis")
assi[1].set_title("gruppi del k-means")
plt.show()

Completare: contare quanti pinguini finiscono nel gruppo in cui la loro specie è in maggioranza. `max(axis=1)` prende il massimo di ogni riga della tabella.

In [ ]:
tabella = pd.crosstab(pinguini["specie"], pinguini["gruppo"])
concordi = tabella.___(axis=1).sum()
print(concordi, "su", len(pinguini), "->", round(concordi / len(pinguini), 3))
controlla(concordi > 300)

## 3. L'effetto della standardizzazione (esercizio 3)

Ripetere senza standardizzare, con le quattro misure (massa in grammi compresa).

In [ ]:
misure = ["becco_lunghezza_mm", "becco_profondita_mm", "pinna_lunghezza_mm", "massa_g"]
completi = pinguini.dropna(subset=misure)
for standardizza in [False, True]:
    Xm = completi[misure].to_numpy()
    if standardizza:
        Xm = StandardScaler().fit_transform(Xm)
    g = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xm).labels_
    t = pd.crosstab(completi["specie"], g)
    print("standardizzato" if standardizza else "grezzo", "-> concordi:", t.max(axis=1).sum(), "su", len(completi))
    print(t, "\n")

**Domanda**: perché senza standardizzazione i gruppi non corrispondono alle specie? (Vedi L8.)

Risposta:

## 4. Quanti gruppi? Il metodo del gomito (esercizio 4)

- `inertia_`: somma dei quadrati delle distanze dei punti dal proprio centroide; diminuisce sempre aumentando k
- si cerca il "gomito": il k oltre il quale la diminuzione diventa piccola

In [ ]:
inerzie = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), inerzie, marker="o")
plt.xlabel("k"); plt.ylabel("inerzia")
plt.show()
print(np.round(inerzie, 1))

## 5. Ridurre i colori di un'immagine (esercizio 5)

Ogni pixel è un punto con tre coordinate (rosso, verde, blu, da 0 a 1). Il k-means raggruppa i pixel in k gruppi di colori simili; ogni pixel viene sostituito con il colore del proprio centroide.

- `np.load`: carica l'immagine salvata come array NumPy (altezza x larghezza x 3)
- `reshape(-1, 3)`: trasforma l'immagine in una tabella con una riga per pixel e tre colonne

In [ ]:
foto = np.load("L15_tramonto.npy")
pixel = foto.reshape(-1, 3)
print(foto.shape, "->", pixel.shape)

k = ___
kmc = KMeans(n_clusters=k, n_init=4, random_state=0).fit(pixel)
ridotta = kmc.cluster_centers_[kmc.labels_].reshape(foto.shape)

fig, assi = plt.subplots(1, 2, figsize=(10, 3.5))
assi[0].imshow(foto); assi[0].set_title("originale"); assi[0].axis("off")
assi[1].imshow(ridotta); assi[1].set_title(f"{k} colori"); assi[1].axis("off")
plt.show()
controlla(len(kmc.cluster_centers_) >= 2, "Scegliere un numero di colori k di almeno 2.")

Provare k = 2, 3, 8, 16. Quanti colori servono perché l'immagine sembri quella originale?